# SVM (LinearSVC + TF-IDF), paper-aligned protocol
The paper lists SVM as a baseline (Table 2) but gives **no features or hyperparameters**, so the TF-IDF/LinearSVC setup is kept from your version.

Aligned with the paper: 70/15/15 stratified split from `train/val/test.jsonl`, seeds 42/123/2025 (mean ± std), macro metrics, bootstrap 95% CI.

Note: LinearSVC is nearly deterministic, so the three seeds will give almost identical scores (std ≈ 0).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, gc, numpy as np, pandas as pd

# ---- EDIT: Drive folder that holds gujarati_fake_real_combined.jsonl and the split files ----
DATA_DIR = "/content/drive/MyDrive/RM_24DCS041_054"
MODEL_DIR = f"{DATA_DIR}/models_paper"      # separate folder so earlier models are not overwritten
os.makedirs(MODEL_DIR, exist_ok=True)

SEEDS = [42, 123, 2025]   # paper (Sec. 4.4/5.1): three seeds, results reported as mean +/- std

In [ ]:
def load_split(name):
    d = pd.read_json(f"{DATA_DIR}/{name}.jsonl", lines=True, dtype={"news_id": str})
    assert {"news_id", "text", "label"} <= set(d.columns)
    d["text"] = d["text"].astype(str)
    d["label"] = d["label"].astype(str).str.strip().str.lower()
    assert d["label"].isin(["fake", "real"]).all()
    return d

train_df, val_df, test_df = load_split("train"), load_split("val"), load_split("test")

# one label encoding for every model: real = 0, fake = 1
label2id = {"real": 0, "fake": 1}
id2label = {0: "real", 1: "fake"}

X_train, y_train = train_df["text"].values, train_df["label"].map(label2id).values
X_val,   y_val   = val_df["text"].values,   val_df["label"].map(label2id).values
X_test,  y_test  = test_df["text"].values,  test_df["label"].map(label2id).values

n = len(train_df) + len(val_df) + len(test_df)
print(f"train {len(train_df)} ({len(train_df)/n:.1%}) | val {len(val_df)} ({len(val_df)/n:.1%}) | test {len(test_df)} ({len(test_df)/n:.1%})")
print("Expected ~70/15/15. If not, re-run 01_data_split.ipynb first.")
print(train_df["label"].value_counts().to_dict())

In [ ]:
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support, f1_score,
                             classification_report, ConfusionMatrixDisplay)
import matplotlib.pyplot as plt

def evaluate(y_true, y_pred, n_boot=1000, seed=0):
    """Macro metrics + bootstrap 95% CI for macro-F1 (paper Sec. 5.1)."""
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    rng = np.random.default_rng(seed)
    boots = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        boots.append(f1_score(y_true[idx], y_pred[idx], average="macro", zero_division=0))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return {"accuracy": accuracy_score(y_true, y_pred), "macro_precision": p,
            "macro_recall": r, "macro_f1": f, "f1_ci_low": lo, "f1_ci_high": hi}

In [ ]:
import joblib, time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

tfidf = TfidfVectorizer(lowercase=False, ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)
Xtr = tfidf.fit_transform(X_train)          # fit on train only
Xva, Xte = tfidf.transform(X_val), tfidf.transform(X_test)
print("TF-IDF:", Xtr.shape, "| vocabulary:", len(tfidf.vocabulary_))

results, all_preds = [], {}
for seed in SEEDS:
    t = time.time()
    clf = LinearSVC(C=1.0, max_iter=5000, random_state=seed).fit(Xtr, y_train)
    pred = clf.predict(Xte)
    m = evaluate(y_test, pred)
    m.update(seed=seed, val_macro_f1=f1_score(y_val, clf.predict(Xva), average="macro"))
    results.append(m); all_preds[seed] = pred
    print(f"seed {seed}: test macro-F1 {m['macro_f1']:.4f} | val macro-F1 {m['val_macro_f1']:.4f} | {time.time()-t:.1f}s")
    if seed == SEEDS[0]:
        joblib.dump({"tfidf": tfidf, "clf": clf}, f"{MODEL_DIR}/03_svm_seed{seed}.joblib")

In [ ]:
res = pd.DataFrame(results).set_index("seed")
display(res.round(4))
cols = ["accuracy", "macro_precision", "macro_recall", "macro_f1"]
summ = res[cols].agg(["mean", "std"]).T
print("=== TEST RESULTS, mean +/- std over seeds " + str(list(res.index)) + " ===")
for c in cols:
    print(f"{c:16s}: {summ.loc[c, 'mean']:.4f} +/- {summ.loc[c, 'std']:.4f}")
res.to_csv(f"{MODEL_DIR}/03_svm_results.csv")

first = SEEDS[0]
print(f"\nClassification report (seed {first}):")
print(classification_report(y_test, all_preds[first], target_names=["real", "fake"], digits=4, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, all_preds[first], display_labels=["real", "fake"], cmap="Blues")
plt.title(f"03_svm - confusion matrix (seed {first})"); plt.show()